In [7]:
import pandas as pd
import os
from pathlib import Path
import glob

In [ ]:
def combinar_atendimentos():
    pasta_com_urgencia = Path("./dados/atendimentos_com_urgencia")
    pasta_sem_urgencia = Path("./dados/atendimentos_sem_urgencia")
    pasta_saida = Path("./dados/")
    
    pasta_saida.mkdir(exist_ok=True)
    
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    
    
    for ano in anos:
        arquivo_com_urgencia = pasta_com_urgencia / f"{ano}.xlsx"
        arquivo_sem_urgencia = pasta_sem_urgencia / f"{ano}.xlsx"
        
        if not arquivo_com_urgencia.exists():
            print(f"  Arquivo não encontrado: {arquivo_com_urgencia}")
            continue
            
        if not arquivo_sem_urgencia.exists():
            print(f"  Arquivo não encontrado: {arquivo_sem_urgencia}")
            continue
        
        try:
            df_com_urgencia = pd.read_excel(arquivo_com_urgencia, sheet_name='Dados', decimal=',', thousands='.')
            
            df_sem_urgencia = pd.read_excel(arquivo_sem_urgencia, sheet_name='Dados', decimal=',', thousands='.')
            
            if list(df_com_urgencia.columns) != list(df_sem_urgencia.columns):
                colunas_comuns = list(set(df_com_urgencia.columns) & set(df_sem_urgencia.columns))
                df_com_urgencia = df_com_urgencia[colunas_comuns]
                df_sem_urgencia = df_sem_urgencia[colunas_comuns]
            colunas_id = ['Uf', 'Ibge', 'Municipio']
            colunas_numericas = [col for col in df_com_urgencia.columns if col not in colunas_id]
            
            for col in colunas_numericas:
                df_com_urgencia[col] = pd.to_numeric(df_com_urgencia[col], errors='coerce').fillna(0)
                df_sem_urgencia[col] = pd.to_numeric(df_sem_urgencia[col], errors='coerce').fillna(0)
            
            df_combinado = pd.merge(
                df_com_urgencia, 
                df_sem_urgencia, 
                on=colunas_id, 
                how='outer',
                suffixes=('_com_urgencia', '_sem_urgencia')
            )
            
            df_resultado = df_combinado[colunas_id].copy()
            
            for col in colunas_numericas:
                col_com = f"{col}_com_urgencia"
                col_sem = f"{col}_sem_urgencia"
                
                if col_com in df_combinado.columns and col_sem in df_combinado.columns:
                    df_resultado[col] = (
                        df_combinado[col_com].fillna(0) + 
                        df_combinado[col_sem].fillna(0)
                    )
                elif col_com in df_combinado.columns:
                    df_resultado[col] = df_combinado[col_com].fillna(0)
                elif col_sem in df_combinado.columns:
                    df_resultado[col] = df_combinado[col_sem].fillna(0)
                else:
                    df_resultado[col] = 0
            
            df_resultado = df_resultado.sort_values(['Uf', 'Municipio'])
            
            arquivo_saida = pasta_saida / f"relatorio_{ano}.xlsx"
            
            with pd.ExcelWriter(arquivo_saida, engine='openpyxl') as writer:
                df_resultado.to_excel(writer, sheet_name='Dados', index=False)
            
            total_municipios = len(df_resultado)
            total_com_urgencia = len(df_com_urgencia)
            total_sem_urgencia = len(df_sem_urgencia)
            principais_doencas = ['Diabetes', 'Hipertensão arterial', 'Saúde mental']
            for doenca in principais_doencas:
                if doenca in df_resultado.columns:
                    total = df_resultado[doenca].sum()
                    print(f"    Total {doenca}: {total:,}")
            
        except Exception as e:
            continue
    
    criar_resumo_geral(pasta_saida)

In [9]:
def criar_resumo_geral(pasta_saida):
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    dados_resumo = []
    
    for ano in anos:
        arquivo = pasta_saida / f"relatorio_{ano}.xlsx"
        if arquivo.exists():
            try:
                df = pd.read_excel(arquivo, sheet_name='Dados')
                colunas_id = ['Uf', 'Ibge', 'Municipio']
                colunas_numericas = [col for col in df.columns if col not in colunas_id]
                
                resumo_ano = {'Ano': ano}
                resumo_ano['Total_Municipios'] = len(df)
                
                for col in colunas_numericas:
                    if col in df.columns:
                        resumo_ano[f'Total_{col}'] = df[col].sum()
                
                dados_resumo.append(resumo_ano)
                
            except Exception as e:
                print(f"  Erro ao processar resumo do ano {ano}: {str(e)}")
    
    if dados_resumo:
        df_resumo = pd.DataFrame(dados_resumo)
        arquivo_resumo = pasta_saida / "resumo_geral_todos_anos.xlsx"
        
        with pd.ExcelWriter(arquivo_resumo, engine='openpyxl') as writer:
            df_resumo.to_excel(writer, sheet_name='Resumo_Geral', index=False)
        
        principais_doencas = ['Total_Diabetes', 'Total_Hipertensão arterial', 'Total_Saúde mental']
        
        for _, row in df_resumo.iterrows():
            print(f"  {row['Ano']}: {row['Total_Municipios']} municípios")
            for doenca in principais_doencas:
                if doenca in row:
                    print(f"    {doenca.replace('Total_', '')}: {row[doenca]:,}")

def gerar_consolidado_com_urgencia():
    pasta_com_urgencia = Path("./dados/atendimentos_com_urgencia")
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    
    dfs_urgencia = []
    
    for ano in anos:
        arquivo = pasta_com_urgencia / f"{ano}.xlsx"
        if arquivo.exists():
            try:
                df = pd.read_excel(arquivo, sheet_name='Dados', decimal=',', thousands='.')
                df['Ano'] = ano
                dfs_urgencia.append(df)
                print(f"  Carregado: {ano} - {len(df)} registros")
            except Exception as e:
                print(f"  Erro ao processar {ano} com urgência: {e}")
    
    if dfs_urgencia:
        merged_urgencia = pd.concat(dfs_urgencia, ignore_index=True)
        
        disease_columns_urgencia = [
            'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
            'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
            'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
            'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
            'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
            'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
        ]
        
        for col in disease_columns_urgencia:
            if col in merged_urgencia.columns:
                merged_urgencia[col] = pd.to_numeric(merged_urgencia[col], errors='coerce')
        
        aggregated_urgencia = merged_urgencia.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns_urgencia if col in merged_urgencia.columns}).reset_index()
        aggregated_urgencia['Ano'] = aggregated_urgencia['Ano'].astype(int)
        
        df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')
        
        df_ibge_long = pd.melt(
            df_ibge,
            id_vars=['Cód.', 'Unidade da Federação'],
            value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
            var_name='Ano', value_name='populacao_ibge'
        )
        
        df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
        df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
        df_proj_2025['Ano'] = 2025
        
        df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
        df_ibge_long = df_ibge_long.rename(columns={
            'Cód.': 'cod_uf',
            'Unidade da Federação': 'nome_uf'
        })
        
        df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)
        
        mask_ate_2024 = df_ibge_long['Ano'] <= 2024
        df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000
        
        uf_mapping = {
            'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
            'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
            'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
            'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
            'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
        }
        df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)
        df_merged_urgencia = pd.merge(
            aggregated_urgencia,
            df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
            on=['Uf', 'Ano'], how='left'
        )
        
        df_merged_urgencia.to_excel('./dados/consolidado_final_com_urgencia.xlsx', index=False)
        print("✅ Arquivo consolidado_final_com_urgencia.xlsx gerado com sucesso!")
        print(f"   Total de registros: {len(df_merged_urgencia)}")

def gerar_consolidado_sem_urgencia():
    pasta_sem_urgencia = Path("./dados/atendimentos_sem_urgencia")
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    
    dfs_sem_urgencia = []
    
    for ano in anos:
        arquivo = pasta_sem_urgencia / f"{ano}.xlsx"
        if arquivo.exists():
            try:
                df = pd.read_excel(arquivo, sheet_name='Dados', decimal=',', thousands='.')
                df['Ano'] = ano
                dfs_sem_urgencia.append(df)
                print(f"  Carregado: {ano} - {len(df)} registros")
            except Exception as e:
                print(f"  Erro ao processar {ano} sem urgência: {e}")
    
    if dfs_sem_urgencia:
        merged_sem_urgencia = pd.concat(dfs_sem_urgencia, ignore_index=True)
        
        disease_columns_sem_urgencia = [
            'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
            'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
            'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
            'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
            'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
            'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
        ]
        
        for col in disease_columns_sem_urgencia:
            if col in merged_sem_urgencia.columns:
                merged_sem_urgencia[col] = pd.to_numeric(merged_sem_urgencia[col], errors='coerce')
        
        aggregated_sem_urgencia = merged_sem_urgencia.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns_sem_urgencia if col in merged_sem_urgencia.columns}).reset_index()
        aggregated_sem_urgencia['Ano'] = aggregated_sem_urgencia['Ano'].astype(int)
        
        df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')
        
        df_ibge_long = pd.melt(
            df_ibge,
            id_vars=['Cód.', 'Unidade da Federação'],
            value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
            var_name='Ano', value_name='populacao_ibge'
        )
        
        df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
        df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
        df_proj_2025['Ano'] = 2025
        
        df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
        df_ibge_long = df_ibge_long.rename(columns={
            'Cód.': 'cod_uf',
            'Unidade da Federação': 'nome_uf'
        })
        
        df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)
        
        mask_ate_2024 = df_ibge_long['Ano'] <= 2024
        df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000
        
        uf_mapping = {
            'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
            'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
            'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
            'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
            'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
        }
        df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)
        
        df_merged_sem_urgencia = pd.merge(
            aggregated_sem_urgencia,
            df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
            on=['Uf', 'Ano'], how='left'
        )
        
        df_merged_sem_urgencia.to_excel('./dados/consolidado_final_sem_urgencia.xlsx', index=False)

def adicionar_dependentes_sus():
    ans_file = '../ANS/merged_ans_populacao.xlsx'
    urgencia_file = './dados/consolidado_final_com_urgencia.xlsx'
    sem_urgencia_file = './dados/consolidado_final_sem_urgencia.xlsx'
    
    try:
        df_ans = pd.read_excel(ans_file)
        df_ans_merge = df_ans[['ano', 'uf', 'dependentes_do_sus', 'percentual_dependentes']].copy()
        df_ans_merge['uf'] = df_ans_merge['uf'].str.upper()
        
        df_urgencia = pd.read_excel(urgencia_file)
        df_sem_urgencia = pd.read_excel(sem_urgencia_file)
        df_urgencia_merged = pd.merge(
            df_urgencia,
            df_ans_merge,
            left_on=['Uf', 'Ano'],
            right_on=['uf', 'ano'],
            how='left'
        )
        df_urgencia_merged = df_urgencia_merged.drop(['uf', 'ano'], axis=1)
        df_sem_urgencia_merged = pd.merge(
            df_sem_urgencia,
            df_ans_merge,
            left_on=['Uf', 'Ano'],
            right_on=['uf', 'ano'],
            how='left'
        )
        df_sem_urgencia_merged = df_sem_urgencia_merged.drop(['uf', 'ano'], axis=1)
        urgencia_matches = df_urgencia_merged['dependentes_do_sus'].notna().sum()
        sem_urgencia_matches = df_sem_urgencia_merged['dependentes_do_sus'].notna().sum()
        df_urgencia_merged.to_excel(urgencia_file, index=False)
        df_sem_urgencia_merged.to_excel(sem_urgencia_file, index=False)
        
        
    except FileNotFoundError as e:
        print("erro")
    except Exception as e:
        print("erro")

if __name__ == "__main__":
    combinar_atendimentos()
    
    files = [f for f in glob.glob('./dados/relatorio_*.xlsx') 
             if 'combinado' not in f and 'populacao' not in f and 'doencas' not in f]
    
    dfs = []
    
    for file in files:
        year = file.split('_')[-1].split('.')[0]
        df = pd.read_excel(file, decimal=',', thousands='.')
        df['Ano'] = year
        dfs.append(df)
    
    merged = pd.concat(dfs, ignore_index=True)
    
    disease_columns = [
        'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
        'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
        'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
        'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
        'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
        'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
    ]
    
    for col in disease_columns:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors='coerce')
    
    aggregated = merged.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns}).reset_index()
    df_sisab = aggregated
    df_sisab['Ano'] = df_sisab['Ano'].astype(int)
    df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')
    
    df_ibge_long = pd.melt(
        df_ibge,
        id_vars=['Cód.', 'Unidade da Federação'],
        value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
        var_name='Ano', value_name='populacao_ibge'
    )
    
    df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
    df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
    df_proj_2025['Ano'] = 2025
    
    df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
    df_ibge_long = df_ibge_long.rename(columns={
        'Cód.': 'cod_uf',
        'Unidade da Federação': 'nome_uf'
    })
    
    df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)
    
    mask_ate_2024 = df_ibge_long['Ano'] <= 2024
    df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000
    
    uf_mapping = {
        'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
        'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
        'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
        'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
        'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
    }
    df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)
    
    print(df_ibge_long.head())
    
    disease_columns = [
        'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
        'Saúde sexual e reprodutiva',
        'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
        'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
        'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
        'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
    ]
    
    df_sisab_agg = df_sisab.groupby(['Uf', 'Ano'])[disease_columns].sum().reset_index()
    
    df_merged = pd.merge(
        df_sisab_agg,
        df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
        on=['Uf', 'Ano'], how='left'
    )
    
    gerar_consolidado_com_urgencia()
    gerar_consolidado_sem_urgencia()
    
    adicionar_dependentes_sus()
    
    df_merged.to_excel('./dados/consolidado_final.xlsx', index=False)

    Total Diabetes: 6,865,023.0
    Total Hipertensão arterial: 16,639,029.0
    Total Saúde mental: 4,356,212.0


/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


    Total Diabetes: 7,566,695.0
    Total Hipertensão arterial: 16,052,388.0
    Total Saúde mental: 4,239,655.0
    Total Diabetes: 12,633,555.0
    Total Hipertensão arterial: 26,288,445.0
    Total Saúde mental: 4,636,537.0
    Total Diabetes: 17,748,592.0
    Total Hipertensão arterial: 36,471,333.0
    Total Saúde mental: 5,251,531.0
    Total Diabetes: 18,185,663.0
    Total Hipertensão arterial: 37,088,172.0
    Total Saúde mental: 5,454,718.0
    Total Diabetes: 10,490,708.0
    Total Hipertensão arterial: 21,059,063.0
    Total Saúde mental: 3,276,964.0
  2019: 5552 municípios
    Diabetes: 6,865,023
    Hipertensão arterial: 16,639,029
    Saúde mental: 4,356,212
  2021: 5566 municípios
    Diabetes: 7,566,695
    Hipertensão arterial: 16,052,388
    Saúde mental: 4,239,655
  2022: 5569 municípios
    Diabetes: 12,633,555
    Hipertensão arterial: 26,288,445
    Saúde mental: 4,636,537
  2023: 5569 municípios
    Diabetes: 17,748,592
    Hipertensão arterial: 36,471,333
    S

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Carregado: 2020 - 3970 registros
  Carregado: 2021 - 3822 registros
  Carregado: 2022 - 4091 registros
  Carregado: 2023 - 4228 registros
  Carregado: 2024 - 4044 registros
  Carregado: 2025 - 3523 registros
✅ Arquivo consolidado_final_com_urgencia.xlsx gerado com sucesso!
   Total de registros: 162
  Carregado: 2019 - 5552 registros
  Carregado: 2020 - 5558 registros
  Carregado: 2021 - 5566 registros
  Carregado: 2022 - 5569 registros
  Carregado: 2023 - 5569 registros
  Carregado: 2024 - 5569 registros
  Carregado: 2025 - 5564 registros
erro


In [10]:
def gerar_consolidado_com_urgencia():
    pasta_com_urgencia = Path("./dados/atendimentos_com_urgencia")
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    
    dfs_urgencia = []
    
    for ano in anos:
        arquivo = pasta_com_urgencia / f"{ano}.xlsx"
        if arquivo.exists():
            try:
                df = pd.read_excel(arquivo, sheet_name='Dados', decimal=',', thousands='.')
                df['Ano'] = ano
                dfs_urgencia.append(df)
                print(f"  Carregado: {ano} - {len(df)} registros")
            except Exception as e:
                print(f"  Erro ao processar {ano} com urgência: {e}")
    
    if dfs_urgencia:
        merged_urgencia = pd.concat(dfs_urgencia, ignore_index=True)
        
        # Processar colunas numéricas
        disease_columns_urgencia = [
            'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
            'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
            'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
            'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
            'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
            'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
        ]
        
        for col in disease_columns_urgencia:
            if col in merged_urgencia.columns:
                merged_urgencia[col] = pd.to_numeric(merged_urgencia[col], errors='coerce')
        
        aggregated_urgencia = merged_urgencia.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns_urgencia if col in merged_urgencia.columns}).reset_index()
        aggregated_urgencia['Ano'] = aggregated_urgencia['Ano'].astype(int)
        
        df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')
        
        df_ibge_long = pd.melt(
            df_ibge,
            id_vars=['Cód.', 'Unidade da Federação'],
            value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
            var_name='Ano', value_name='populacao_ibge'
        )
        df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
        df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
        df_proj_2025['Ano'] = 2025
        
        df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
        df_ibge_long = df_ibge_long.rename(columns={
            'Cód.': 'cod_uf',
            'Unidade da Federação': 'nome_uf'
        })
        
        df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)
        
        mask_ate_2024 = df_ibge_long['Ano'] <= 2024
        df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000
        
        uf_mapping = {
            'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
            'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
            'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
            'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
            'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
        }
        df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)
        df_merged_urgencia = pd.merge(
            aggregated_urgencia,
            df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
            on=['Uf', 'Ano'], how='left'
        )
        
        print("Consolidado com urgência:")
        print(f"Total de registros: {len(df_merged_urgencia)}")
        print(df_merged_urgencia.head())

gerar_consolidado_com_urgencia()

  Carregado: 2019 - 4169 registros


/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  Carregado: 2020 - 3970 registros
  Carregado: 2021 - 3822 registros
  Carregado: 2022 - 4091 registros
  Carregado: 2023 - 4228 registros
  Carregado: 2024 - 4044 registros
  Carregado: 2025 - 3523 registros
Consolidado com urgência:
Total de registros: 162
   Uf   Ano  Asma  Desnutrição  Diabetes  DPOC  Hipertensão arterial  \
0  AC  2019   3.0          7.0      34.0   7.0                 119.0   
1  AC  2021   1.0          5.0      30.0   2.0                  88.0   
2  AC  2022   0.0          2.0      25.0   3.0                  61.0   
3  AC  2023   1.0          4.0      33.0   3.0                  83.0   
4  AC  2024   0.0          3.0      28.0   8.0                  96.0   

   Obesidade  Pré-natal  Puericultura  ...  Saúde mental  Reabilitação  \
0        3.0        0.0           0.0  ...          12.0          34.0   
1        6.0        0.0           2.0  ...          11.0          11.0   
2        5.0        0.0           2.0  ...           7.0           4.0   
3        3.

In [11]:
def gerar_consolidado_sem_urgencia():
    pasta_sem_urgencia = Path("./dados/atendimentos_sem_urgencia")
    anos = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']
    
    dfs_sem_urgencia = []
    
    for ano in anos:
        arquivo = pasta_sem_urgencia / f"{ano}.xlsx"
        if arquivo.exists():
            try:
                df = pd.read_excel(arquivo, sheet_name='Dados', decimal=',', thousands='.')
                df['Ano'] = ano
                dfs_sem_urgencia.append(df)
                print(f"  Carregado: {ano} - {len(df)} registros")
            except Exception as e:
                print(f"  Erro ao processar {ano} sem urgência: {e}")
    
    if dfs_sem_urgencia:
        merged_sem_urgencia = pd.concat(dfs_sem_urgencia, ignore_index=True)
        
        disease_columns_sem_urgencia = [
            'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
            'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
            'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
            'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
            'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
            'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
        ]
        
        for col in disease_columns_sem_urgencia:
            if col in merged_sem_urgencia.columns:
                merged_sem_urgencia[col] = pd.to_numeric(merged_sem_urgencia[col], errors='coerce')
        
        aggregated_sem_urgencia = merged_sem_urgencia.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns_sem_urgencia if col in merged_sem_urgencia.columns}).reset_index()
        aggregated_sem_urgencia['Ano'] = aggregated_sem_urgencia['Ano'].astype(int)
        
        df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')
        
        df_ibge_long = pd.melt(
            df_ibge,
            id_vars=['Cód.', 'Unidade da Federação'],
            value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
            var_name='Ano', value_name='populacao_ibge'
        )
        
        df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
        df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
        df_proj_2025['Ano'] = 2025
        
        df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
        df_ibge_long = df_ibge_long.rename(columns={
            'Cód.': 'cod_uf',
            'Unidade da Federação': 'nome_uf'
        })
        
        df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)
        
        mask_ate_2024 = df_ibge_long['Ano'] <= 2024
        df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000
        
        uf_mapping = {
            'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
            'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
            'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
            'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
            'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
        }
        df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)
        
        df_merged_sem_urgencia = pd.merge(
            aggregated_sem_urgencia,
            df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
            on=['Uf', 'Ano'], how='left'
        )
        
gerar_consolidado_sem_urgencia()

  Carregado: 2019 - 5552 registros
  Carregado: 2020 - 5558 registros
  Carregado: 2021 - 5566 registros
  Carregado: 2022 - 5569 registros
  Carregado: 2023 - 5569 registros
  Carregado: 2024 - 5569 registros
  Carregado: 2025 - 5564 registros


In [12]:
files = [f for f in glob.glob('./dados/relatorio_*.xlsx') 
         if 'combinado' not in f and 'populacao' not in f and 'doencas' not in f]

dfs = []

for file in files:
    year = file.split('_')[-1].split('.')[0]
    df = pd.read_excel(file, decimal=',', thousands='.')
    df['Ano'] = year
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

disease_columns = [
    'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
    'Pré-natal', 'Puericultura', 'Puerpério (até 42 dias)', 'Saúde sexual e reprodutiva',
    'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
    'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
    'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
    'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
]

for col in disease_columns:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors='coerce')

aggregated = merged.groupby(['Uf', 'Ano']).agg({col: 'sum' for col in disease_columns}).reset_index()
df_sisab = aggregated
df_sisab['Ano'] = df_sisab['Ano'].astype(int)

df_ibge = pd.read_excel('../IBGE/populacao_PNAD.xlsx')

df_ibge_long = pd.melt(
    df_ibge,
    id_vars=['Cód.', 'Unidade da Federação'],
    value_vars=[2019, 2020, 2021, 2022, 2023, 2024],
    var_name='Ano', value_name='populacao_ibge'
)

df_proj_2025 = pd.read_excel('../IBGE/projecao_2025.xlsx')
df_proj_2025 = df_proj_2025.rename(columns={'total': 'populacao_ibge'})
df_proj_2025['Ano'] = 2025

df_ibge_long = pd.concat([df_ibge_long, df_proj_2025], ignore_index=True)
df_ibge_long = df_ibge_long.rename(columns={
    'Cód.': 'cod_uf',
    'Unidade da Federação': 'nome_uf'
})

df_ibge_long['Ano'] = df_ibge_long['Ano'].astype(int)

mask_ate_2024 = df_ibge_long['Ano'] <= 2024
df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] = df_ibge_long.loc[mask_ate_2024, 'populacao_ibge'] * 1000

uf_mapping = {
    'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP', 'Tocantins': 'TO',
    'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Pernambuco': 'PE',
    'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ',
    'São Paulo': 'SP', 'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
    'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'
}
df_ibge_long['Uf'] = df_ibge_long['nome_uf'].map(uf_mapping)


disease_columns = [
    'Asma', 'Desnutrição', 'Diabetes', 'DPOC', 'Hipertensão arterial', 'Obesidade',
    'Saúde sexual e reprodutiva',
    'Tabagismo', 'Usuário de álcool', 'Usuário de outras drogas', 'Saúde mental',
    'Reabilitação', 'D.Transmissíveis - Dengue', 'Doenças transmissíveis - DST',
    'D.Transmissíveis - Hanseníase', 'D.Transmissíveis - Tuberculose',
    'Rast. câncer de mama', 'Rast. câncer do colo do útero', 'Rast. risco cardiovascular'
]

df_sisab_agg = df_sisab.groupby(['Uf', 'Ano'])[disease_columns].sum().reset_index()

df_merged = pd.merge(
    df_sisab_agg,
    df_ibge_long[['Uf', 'Ano', 'populacao_ibge']],
    on=['Uf', 'Ano'], how='left'
)